# FastLane small models: QLoRA on Qwen2.5-3B (Kaggle, both adapters in one run) — data v5

Trains **two LoRA adapters** back to back on one T4:
- `all_domains` → the FastLane small model (used by FastLane and most variants), ~3.3 h
- `movies_buses` → the new-domain test adapter for variant A8 (no restaurant data), ~2.3 h

**Data:** `runs/data_v5/colab_bundle.zip` (md5 `406c9dd868fbd58277e73538058729fe`; same file as
`data/train/colab_bundle.zip`), uploaded as a **new version** of the **fastlane-train** dataset.

**Setup:** *Add Input* → fastlane-train (**latest version**), *Accelerator* → GPU T4 x2, *Internet* → On.
Run cells 1–3 first: cell 3 checks the data and **stops with an error if it is not v5**
(v5: all_domains 3153 / 245, movies_buses 2243 / 146; v3 was 3671 / 267, v4 3337 / 245).
Then *Save Version* → **Save & Run All (Commit)** and close the tab.

**Result:** in the version's *Output* tab, download `fastlane-qwen2.5-3b-adapter.zip` and
`fastlane-qwen2.5-3b-mb-adapter.zip` (and the log) into `kaggle_output_6/` on your machine.

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTERS = {"all_domains": "fastlane-qwen2.5-3b", "movies_buses": "fastlane-qwen2.5-3b-mb"}
# Data v5 changes both data sets, and A8 must be trained on the same data version as FastLane for a fair
# new-domain comparison, so this run trains both (~5.5 h).
TRAIN = ["all_domains", "movies_buses"]
EPOCHS = 2
# Expected (train, val) rows of data v5. Cell 3 stops the run if the attached dataset differs
# (the run in kaggle_output_5 trained v3 by mistake because the old dataset version was attached).
EXPECTED = {"all_domains": (3153, 245), "movies_buses": (2243, 146)}
EXPECTED_NOISE = "roundtrip"

In [ ]:
!pip -q install -U "transformers>=4.45" "peft>=0.12" "trl>=0.20" "datasets>=2.20" "bitsandbytes>=0.43" accelerate
# Kaggle ships an old torchao that breaks peft; it is not needed.
!pip -q uninstall -y torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, json, os, subprocess
scripts = glob.glob("/kaggle/input/**/train_lora.py", recursive=True)
roots = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/all_domains/train.jsonl", recursive=True)})
print("train_lora.py:", scripts)
print("data roots:", roots)
assert len(roots) == 1, f"expected exactly one attached data folder, found {roots}: remove old inputs"
script, data_root = scripts[0], roots[0]

for a in TRAIN:
    n_train = sum(1 for _ in open(f"{data_root}/{a}/train.jsonl"))
    n_val = sum(1 for _ in open(f"{data_root}/{a}/val.jsonl"))
    noise = json.load(open(f"{data_root}/{a}/stats.json")).get("noise")
    print(a, n_train, "train /", n_val, "val, noise =", noise)
    assert (n_train, n_val) == EXPECTED[a], (
        f"{a}: got {n_train}/{n_val}, expected {EXPECTED[a]} (data v5). "
        "Wrong dataset version attached: Input panel -> fastlane-train -> latest version.")
    assert noise == EXPECTED_NOISE, f"{a}: noise={noise!r}, expected {EXPECTED_NOISE!r} (data v5)"
print("OK: data v5 attached")

In [ ]:
# Train each adapter (loss on the target JSON only) and zip it straight away, so finished adapters are kept
# in the Output tab even if a later run fails. check=True stops the notebook on an error.
# The data stats go into the zip too, so the downloaded adapter says which data it was trained on.
import shutil
for adapter in TRAIN:
    name = ADAPTERS[adapter]
    out = f"/kaggle/working/{name}"
    subprocess.run(["python", script, "--data", f"{data_root}/{adapter}", "--model", MODEL, "--out", out,
                    "--epochs", str(EPOCHS), "--batch", "2", "--grad-accum", "8"],
                   check=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
    shutil.copy(f"{data_root}/{adapter}/stats.json", f"{out}/data_stats.json")
    subprocess.run(f"cd /kaggle/working && zip -q -r {name}-adapter.zip {name}/adapter {name}/eval.json {name}/data_stats.json && rm -rf {name}/checkpoint-*",
                   shell=True, check=True)
    print(name, open(f"{out}/eval.json").read())

In [ ]:
!ls -lh /kaggle/working